# PS5 — Hardware-OOS Failure Spine + Per-Serial Component Table (notebook-only)

**What this does.** Rebuilds the PS5 failure event as **any hardware OOS 'Set'** — uniformly for TVM / gates / validators — read *directly* from `silver.device_event_enriched` (the 190.9M-row OOS stream), instead of the chargeable-bounded `silver.device_failures` spine. It also builds a per-serial component table with **true per-component OOS counts** and captures **chargeable** events as an analysis column (device + serial level).

**Guardrails (read them):**
- **Read-only from silver.** It never writes to any `mars_dev.silver.*` or `mars_dev.gold.*` table. Outputs go to a local folder (`PS5_oos_spine_outputs/`) and are wired into the PS5 engine **in memory**.
- **Event = hardware OOS Set**, NOT chargeable. Chargeable (`FAILURE_LEVEL>0 AND EXCLUDED=0` from `kpi_avail_enriched`) is captured as `total_chargeable_events` / `chargeable_pct` for analysis only.
- **Validators (BMV)** have no chargeable stream, so their `total_chargeable_events` is 0 by construction — exactly as expected.
- **Per-serial is counts + age, not a survival curve.** The catalog confirms component-level survival is 100% censored (no observed component replacements), so per-serial reliability here is OOS/chargeable counts + component age + a risk score, matching your `serial_id_device_id.xlsx`. Device-level RUL stays the survival model.
- **Schema-adaptive.** Column names are resolved at runtime from candidate lists and printed, so you can verify the match before trusting the output.

**Run order:** run cells top-to-bottom in the **same kernel as your PS5 engine cell** (so `CONFIG` exists for the wiring cell). Then re-run `main(synthetic=False)`.


In [ ]:
# ── Cell 1 — Spark bootstrap (reuse an active session, else local S3A) ─────────
# Safest: run this in the SAME kernel as your working PS1 validator notebook (its Spark
# session + Java are already configured) — this cell will just reuse it. Otherwise it
# builds a local Spark with S3A, mirroring your PS1 cell-8 config.
import os, sys, subprocess, importlib

def _pip(*pkgs):
    try: ip = get_ipython()
    except NameError: ip = None
    if ip is not None: ip.run_line_magic("pip", "install -q " + " ".join(pkgs))
    else: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    importlib.invalidate_caches()

try:
    import pyspark  # noqa
except ImportError:
    _pip("pyspark==3.5.9")

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

HADOOP_AWS_PACKAGE = "org.apache.hadoop:hadoop-aws:3.3.4"   # matches Spark 3.5 / Hadoop 3.3.x

spark = SparkSession.getActiveSession()
if spark is None:
    # local fallback; needs a JVM. If Java is missing, run this notebook in your PS1
    # kernel or a SageMaker SparkMagic/Glue kernel instead.
    os.environ.setdefault("PYSPARK_SUBMIT_ARGS", "--driver-memory 22g pyspark-shell")
    spark = (SparkSession.builder.appName("PS5_OOS_Spine_Builder")
             .master("local[*]")
             .config("spark.driver.memory", "22g")
             .config("spark.driver.maxResultSize", "6g")
             .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
             .config("spark.sql.adaptive.enabled", "true")
             .config("spark.jars.packages", HADOOP_AWS_PACKAGE)
             .config("spark.hadoop.fs.s3a.aws.credentials.provider",
                     "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
             .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
             .getOrCreate())
print("Spark:", spark.version, "| master:", spark.sparkContext.master)


In [ ]:
# ── Cell 2 — Config (paths, window, scope). Mirrors the PS5 engine. ───────────
GOLD_BUCKET     = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
S3_SILVER       = f"s3://{GOLD_BUCKET}/chicago/silver"
# NOTE: the PS5 engine reads silver parquet via s3a here too. If your Spark reads use
# "s3://" instead of "s3a://", flip the scheme below.
SCHEME          = "s3a"
def SILVER(t):  return f"{SCHEME}://{GOLD_BUCKET}/chicago/silver/{t}"

RUN_DATE        = "2026-04-11"     # ODS watermark; drop events after this (matches engine RUN_DATE)
TELEMETRY_START = "2024-01-01"     # device_event_enriched window start
SCOPE           = ["TVM", "GATE", "VALIDATOR"]

# operational (non-hardware) events to exclude, per the Data Foundation doc
OPS_EXCLUDE = ["emp logon", "in service", "maintenance mode", "vehicle ignition off", "cmdoosbytables"]

OUT_LOCAL = "PS5_oos_spine_outputs"   # LOCAL scratch (NOT a silver/gold table)
os.makedirs(OUT_LOCAL, exist_ok=True)
print("scope:", SCOPE, "| window:", TELEMETRY_START, "->", RUN_DATE, "| out:", OUT_LOCAL)


In [ ]:
# ── Cell 3 — Helpers + schema probe (verify the adaptive column matches) ─────
def resolve(available, wanted):
    """Case-insensitive: return {role: actual_col_or_None} from candidate lists."""
    low = {c.lower(): c for c in available}
    return {role: next((low[c.lower()] for c in cands if c.lower() in low), None)
            for role, cands in wanted.items()}

def truthy(colname):
    c = F.col(colname)
    return (c.cast("boolean") == F.lit(True)) | \
           (F.lower(F.trim(c.cast("string"))).isin("true", "1", "t", "y", "yes"))

def cols_of(table):
    try:
        return spark.read.parquet(SILVER(table)).limit(1).columns
    except Exception as e:
        print(f"  [probe] {table}: {type(e).__name__} {e}"); return []

for t in ["device_event_enriched", "kpi_avail_enriched", "hw_config_current",
          "dim_failure_level", "maintenance_ledger", "dim_event_type", "dim_event_matrix"]:
    c = cols_of(t)
    print(f"\n### {t} ({len(c)} cols)\n" + ", ".join(c))


In [ ]:
# ── Cell 4 — Build the hardware-OOS-Set spine from device_event_enriched ─────
# device_event_enriched is the enriched OOS stream; we resolve the real columns, keep
# fault_state='Set' + counted-as-OOS + device-fault, drop commanded/maintenance/ops,
# window to [TELEMETRY_START, RUN_DATE], and dedup to one episode per device per day.
ev = spark.read.parquet(SILVER("device_event_enriched"))
Rev = resolve(ev.columns, {
    "dev":   ["DEVICE_ID", "device_id"],
    "key":   ["DEVICE_KEY", "device_key"],
    "cat":   ["mars_device_category", "device_category", "DEVICE_CATEGORY"],
    "dtm":   ["EVENT_DTM", "event_dtm", "EVENT_START_DTM", "FAULT_SET_DTM", "START_DTM", "EVENT_TS"],
    "state": ["FAULT_STATE", "fault_state", "AE_FAULT_STATE", "EVENT_FAULT_STATE"],
    "oos":   ["is_counted_oos", "counted_as_oos", "is_oos_event", "is_oos", "is_counted_as_oos", "counted_oos"],
    "devfault": ["is_device_fault", "IS_DEVICE_FAULT"],
    "cmd":   ["is_commanded_oos", "IS_COMMANDED_OOS"],
    "maint": ["is_maintenance_oos", "IS_MAINTENANCE_OOS"],
    "etname":["EVENT_TYPE_NAME", "event_type_name", "EVENT_NAME", "event_description", "EVENT_DESC"],
    "ser":   ["COMPONENT_SERIAL_NBR", "component_serial_nbr"],
})
print("resolved device_event_enriched ->", {k: v for k, v in Rev.items()})
assert Rev["dev"] and Rev["dtm"] and Rev["cat"], "DEVICE_ID / event date / category not found — check the probe above"

applied = []
df = ev.withColumn("_d", F.to_date(F.col(Rev["dtm"])))
df = df.filter((F.col("_d") >= F.lit(TELEMETRY_START)) & (F.col("_d") <= F.lit(RUN_DATE))); applied.append("window")
if Rev["state"]:
    df = df.filter(F.lower(F.trim(F.col(Rev["state"]).cast("string"))).isin("set", "s")); applied.append("fault_state=Set")
if Rev["oos"]:
    df = df.filter(truthy(Rev["oos"])); applied.append(f"{Rev['oos']}=TRUE")
else:
    print("  [warn] no counted-as-OOS flag on device_event_enriched; relying on state+device_fault+ops-exclude. "
          "If dim_event_type carries the flag, add a join here.")
if Rev["devfault"]:
    df = df.filter(truthy(Rev["devfault"])); applied.append(f"{Rev['devfault']}=TRUE")
if Rev["cmd"]:
    df = df.filter(~truthy(Rev["cmd"])); applied.append(f"NOT {Rev['cmd']}")
if Rev["maint"]:
    df = df.filter(~truthy(Rev["maint"])); applied.append(f"NOT {Rev['maint']}")
if Rev["etname"]:
    for nm in OPS_EXCLUDE:
        df = df.filter(~F.lower(F.col(Rev["etname"]).cast("string")).contains(nm))
    applied.append("exclude ops events")
df = df.filter(F.upper(F.col(Rev["cat"]).cast("string")).isin(*SCOPE)); applied.append("scope TVM/GATE/VALIDATOR")
print("filters applied:", applied)

sel = [F.col(Rev["dev"]).cast("string").alias("DEVICE_ID"),
       (F.col(Rev["key"]).cast("string").alias("DEVICE_KEY") if Rev["key"] else F.col(Rev["dev"]).cast("string").alias("DEVICE_KEY")),
       F.upper(F.col(Rev["cat"]).cast("string")).alias("mars_device_category"),
       F.col("_d").alias("failure_date"),
       (F.col(Rev["ser"]).cast("string").alias("COMPONENT_SERIAL_NBR") if Rev["ser"] else F.lit(None).cast("string").alias("COMPONENT_SERIAL_NBR"))]
base = df.select(*sel).persist()

# one hardware-OOS episode per device per day  (== engine FAILURE_EPISODE_DEDUP="day")
epi_dev = base.dropDuplicates(["DEVICE_ID", "failure_date"]).persist()
oos_dev = epi_dev.groupBy("DEVICE_ID", "DEVICE_KEY", "mars_device_category") \
                 .agg(F.count("*").alias("total_hardware_oos_events"))
# TRUE per-component OOS count (day-deduped per component serial)
comp_oos = (base.filter(F.col("COMPONENT_SERIAL_NBR").isNotNull())
                .dropDuplicates(["DEVICE_ID", "COMPONENT_SERIAL_NBR", "failure_date"])
                .groupBy("DEVICE_ID", "COMPONENT_SERIAL_NBR")
                .agg(F.count("*").alias("component_oos_events")))
print("episodes:", epi_dev.count(), "| devices with OOS:", oos_dev.count())
epi_dev.groupBy("mars_device_category").agg(F.count("*").alias("episodes"),
        F.countDistinct("DEVICE_ID").alias("devices")).show()


In [ ]:
# ── Cell 5 — Chargeable device counts from kpi_avail_enriched (analysis only) ─
av = spark.read.parquet(SILVER("kpi_avail_enriched"))
Rav = resolve(av.columns, {
    "dev":  ["DEVICE_ID", "device_id"],
    "key":  ["DEVICE_KEY", "device_key"],
    "fl":   ["FAILURE_LEVEL", "failure_level"],
    "exc":  ["EXCLUDED", "excluded"],
    "state":["FAULT_STATE", "fault_state"],
    "dtm":  ["START_DTM", "start_dtm", "transit_day", "TRANSIT_DAY"],
})
print("resolved kpi_avail_enriched ->", {k: v for k, v in Rav.items()})
ch = av
if Rav["fl"]:    ch = ch.filter(F.col(Rav["fl"]).cast("double") > 0)          # chargeable = FAILURE_LEVEL>0
if Rav["exc"]:   ch = ch.filter((F.col(Rav["exc"]).cast("double") == 0) | F.col(Rav["exc"]).isNull())  # AND not EXCLUDED
if Rav["state"]: ch = ch.filter(F.lower(F.trim(F.col(Rav["state"]).cast("string"))).isin("set", "s"))
if Rav["dtm"]:   ch = ch.withColumn("_cd", F.to_date(F.col(Rav["dtm"]))).filter(F.col("_cd") <= F.lit(RUN_DATE))
else:            ch = ch.withColumn("_cd", F.lit(None))
charge_dev = (ch.dropDuplicates([Rav["dev"], "_cd"])
                .groupBy(F.col(Rav["dev"]).cast("string").alias("DEVICE_ID"))
                .agg(F.count("*").alias("total_chargeable_events")))
print("devices with chargeable:", charge_dev.count())


In [ ]:
# ── Cell 6 — Device summary: OOS + chargeable + chargeable_pct ───────────────
summary = (oos_dev.join(charge_dev, "DEVICE_ID", "left")
                  .fillna({"total_chargeable_events": 0})
                  .withColumn("chargeable_pct",
                              F.round(100.0 * F.col("total_chargeable_events") /
                                      F.greatest(F.col("total_hardware_oos_events"), F.lit(1)), 2))
                  .persist())
print("device summary rows:", summary.count())
summary.groupBy("mars_device_category").agg(
    F.countDistinct("DEVICE_ID").alias("devices"),
    F.sum("total_hardware_oos_events").alias("oos_events"),
    F.sum("total_chargeable_events").alias("chargeable_events"),
    F.round(100.0*F.sum("total_chargeable_events")/F.greatest(F.sum("total_hardware_oos_events"), F.lit(1)), 2).alias("chargeable_pct")
).show()
print("Sanity: TVM/gate chargeable_pct should be ~6-8% (per serial_id_device_id.xlsx); VALIDATOR = 0.")


In [ ]:
# ── Cell 7 — Per-serial component table (hw_config x device summary + age) ────
hw = spark.read.parquet(SILVER("hw_config_current"))
Rhw = resolve(hw.columns, {
    "dev":  ["DEVICE_ID", "device_id"],
    "cat":  ["mars_device_category", "device_category"],
    "ser":  ["COMPONENT_SERIAL_NBR", "component_serial_nbr"],
    "desc": ["COMPONENT_DESCRIPTION", "component_description", "COMPONENT_TYPE_NAME"],
    "age":  ["component_age_days", "COMPONENT_AGE_DAYS"],
    "chg":  ["REPORTED_CHANGED_DTM", "reported_changed_dtm"],
})
print("resolved hw_config_current ->", {k: v for k, v in Rhw.items()})
age_col = (F.col(Rhw["age"]).cast("double") if Rhw["age"]
           else F.datediff(F.lit(RUN_DATE), F.to_date(F.col(Rhw["chg"]))).cast("double"))
serial = (hw.select(
              F.col(Rhw["dev"]).cast("string").alias("DEVICE_ID"),
              F.upper(F.col(Rhw["cat"]).cast("string")).alias("mars_device_category"),
              F.col(Rhw["ser"]).cast("string").alias("COMPONENT_SERIAL_NBR"),
              F.col(Rhw["desc"]).cast("string").alias("COMPONENT_TYPE_NAME"),
              F.greatest(age_col, F.lit(0.0)).alias("component_age_days"))
            .join(summary.select("DEVICE_ID", "total_hardware_oos_events", "total_chargeable_events", "chargeable_pct"),
                  "DEVICE_ID", "left")
            .join(comp_oos, ["DEVICE_ID", "COMPONENT_SERIAL_NBR"], "left")
            .fillna({"total_hardware_oos_events": 0, "total_chargeable_events": 0,
                     "chargeable_pct": 0.0, "component_oos_events": 0})
            .persist())
print("serial rows:", serial.count())
serial.show(6, truncate=False)


In [ ]:
# ── Cell 8 — Write compact outputs to LOCAL scratch (NOT silver/gold) ───────
summary_pd = summary.toPandas()
serial_pd  = serial.toPandas()
epi_pd     = epi_dev.toPandas()

summary_pd.to_parquet(f"{OUT_LOCAL}/device_oos_summary.parquet", index=False)
serial_pd.to_parquet(f"{OUT_LOCAL}/device_serial_oos.parquet", index=False)
epi_pd.to_parquet(f"{OUT_LOCAL}/device_oos_episodes.parquet", index=False)
summary_pd.to_csv(f"{OUT_LOCAL}/device_oos_summary.csv", index=False)
serial_pd.to_csv(f"{OUT_LOCAL}/device_serial_oos.csv", index=False)
print("wrote:", OUT_LOCAL, "-> device_oos_summary / device_serial_oos / device_oos_episodes",
      f"({len(summary_pd)} devices, {len(serial_pd)} serials, {len(epi_pd)} episodes)")
# (Optional) also stage to a SCRATCH S3 prefix — NOT a silver/gold table — if you want it off-box:
# spark.createDataFrame(summary_pd).write.mode("overwrite").parquet(f"s3a://{GOLD_BUCKET}/chicago/ps5/notebook_oos_spine/device_oos_summary")


In [ ]:
# ── Cell 9 — Wire into the PS5 engine (in-memory override; NO table writes) ──
# The engine's load_table() honours a globals()["_SYNTH"] dict: any table present there
# is served from the DataFrame instead of S3. We register ONLY the failure spine and the
# component table, so every OTHER source still loads from silver normally.
# Requires the PS5 engine cell (which defines CONFIG + load_table) to have run in this kernel.
import numpy as np, pandas as pd
assert "CONFIG" in globals(), "Run the PS5 engine cell first (CONFIG must exist)."

# failure spine in the shape load_hw_oos_failures() expects (already OOS-Set; flags pass-through)
oos_engine = epi_pd.copy()
oos_engine["device_category"]    = oos_engine["mars_device_category"]
oos_engine["fault_state"]        = "Set"
oos_engine["is_oos_event"]       = True
oos_engine["is_device_fault"]    = True
oos_engine["is_commanded_oos"]   = False
oos_engine["is_maintenance_oos"] = False
# component table in the shape run_serial_grain() expects
serial_engine = serial_pd.copy()

globals().setdefault("_SYNTH", {})
globals()["_SYNTH"][CONFIG["FAILURE_TABLE"]]   = oos_engine       # replaces silver.device_failures (OOS, all types)
globals()["_SYNTH"][CONFIG["COMPONENT_TABLE"]] = serial_engine    # replaces gold.device_ps5_component (true per-serial)
CONFIG["EVENT_DEF_VERSION"] = "2026-07-24.oos_notebook_spine"     # stamp lineage on the outputs

print("Wired _SYNTH override:")
print("  FAILURE_TABLE  ->", CONFIG["FAILURE_TABLE"], "rows:", len(oos_engine))
print("  COMPONENT_TABLE->", CONFIG["COMPONENT_TABLE"], "rows:", len(serial_engine))
print("Now re-run:  results = main(synthetic=False)   # PS5 will score on hardware-OOS, chargeable captured on the serial table")


## How to run, and what changes

1. Run **Cell 1–8** (Spark). Check the printed **`filters applied`** line and the **`chargeable_pct`** per type — TVM/gates should land ~6–8% (matching `serial_id_device_id.xlsx`), validators 0.
2. Run **Cell 9** in the **same kernel as your PS5 engine** (so `CONFIG`/`load_table` exist). It registers the OOS spine + true per-serial table as in-memory overrides.
3. Re-run `results = main(synthetic=False)`. PS5 now scores on **hardware OOS** for every type; the device counts jump from the chargeable ~7% to the full OOS set, so TVM/gates get far richer survival signal.

**What is and isn't changed**
- **No silver/gold tables are touched.** Outputs live in `PS5_oos_spine_outputs/` and the in-memory `_SYNTH` override.
- **Failure event = hardware OOS.** Chargeable is carried as `total_chargeable_events` + `chargeable_pct` on the device summary and the serial table for analysis — never used as the failure label.
- **Validators** are unaffected in definition (already OOS) but now come through the *same* uniform path.
- **Per-serial = counts + age + risk**, not a survival curve (component-level survival is 100% censored per the catalog). `component_oos_events` is the *true* per-component OOS count; `total_hardware_oos_events` is the device roll-up (matches the xlsx).

**If a probe shows a column I guessed wrong:** the resolver prints what it matched for each source. If `oos` came back `None` on `device_event_enriched`, tell me the real flag column (or the `dim_event_type`/`dim_event_matrix` OOS-flag column) and I'll add the matrix join — everything else will still be correct.
